# 04e. 최종 모델 크기·속도 재측정

- 담당: **④ 경량화·통합** (이 파일은 담당자만 수정)
- 목적: 최종 모델(제출 15, final_solution 3개 앙상블)과 경량 LR 단독의 크기·예측 속도 비교 (서비스 기획 "왜 경량 모델인가" 근거)
- 비교 대상: LR(E + 위치 구간) · 가중치 XGBoost(E) · LightGBM g3 · 최종 3개 앙상블(1:1:1)
- 측정 방법: `submit/final_solution.ipynb`의 피처·모델 정의를 그대로 불러와 train 전체로 학습 → 원본 데이터에서 확률까지 걸리는 시간 (워밍업 1회 후 1명 20회·500명 5회 중앙값), 크기는 저장 크기
- 측정 조건: train 앞 500명 사용 (test 사용 안 함), 같은 컴퓨터 CPU. 환경은 0번 출력
- 성능: 전체 Train OOF Macro F1 (experiments/preds 저장 예측 기준)
- 기록표에는 적지 않음 (결과는 이 노트북 출력), Run All 가능 (Mac 약 15분)

## 0. 환경 설정 (수정하지 말고 그대로 실행)

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/data


In [2]:
import io, time, platform, joblib
import numpy as np, pandas as pd, sklearn

from src.data_loader import load_data, split_xy, encode_target

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)

OS: macOS-27.0.1-arm64-arm-64bit-Mach-O | python 3.13.18 | pandas 3.1.1 | scikit-learn 1.9.1


## 1. 데이터·측정 함수

In [3]:
train, test = load_data(DATA_DIR)   # test는 사용 안 함
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)

def size_mb(*models):
    """저장했을 때의 크기(MB). 모델이 여러 개면 함께 저장한 크기."""
    buffer = io.BytesIO()
    joblib.dump(models if len(models) > 1 else models[0], buffer)
    return round(buffer.getbuffer().nbytes / 1024**2, 2)

def median_ms(fn, n_repeat):
    """워밍업 1회 후 n_repeat번 실행 시간의 중앙값(ms)."""
    fn()
    times = []
    for _ in range(n_repeat):
        start = time.perf_counter()
        fn()
        times.append((time.perf_counter() - start) * 1000)
    return round(float(np.median(times)), 3)

X_timing = X.iloc[:500]   # 시간 측정용 (train, test 사용 안 함)
print("train", X.shape, "| 측정용", X_timing.shape)

train (6201, 4384) | 측정용 (500, 4384)


## 2. 최종 모델 불러오기 — submit/final_solution.ipynb의 【② 피처】·【③ 모델】 칸 그대로

In [ ]:
# 최종 제출 코드(submit/final_solution.ipynb)의 피처·모델 정의를 그대로 불러오기 (같은 모델로 측정)
import json
from pathlib import Path

SEED = 42
final_nb = json.loads((Path(REPO_DIR) / "submit" / "final_solution.ipynb").read_text(encoding="utf-8"))
for marker in ["【② 피처】 시작", "【③ 모델】 시작"]:
    cell = next(c for c in final_nb["cells"] if c["cell_type"] == "code" and marker in "".join(c["source"]))
    exec("".join(cell["source"]))
print("불러온 모델:", list(MODELS))

불러온 모델: ['xgb_E_weighted', 'lr_E_position', 'lgbm_g3']


## 3. train 전체로 학습 (final_solution과 같은 방법)

In [ ]:
# train 전체로 학습 (final_solution과 같은 방법) — Mac 약 10분
fitted = {}
for name, make in MODELS.items():
    t0 = time.time()
    fitted[name] = make().fit(X, y)
    print(name, f"학습 완료 ({(time.time() - t0) / 60:.1f}분)")

xgb_E_weighted 학습 완료 (2.0분)
lr_E_position 학습 완료 (1.2분)
lgbm_g3 학습 완료 (5.2분)


## 4. 모델별 크기·예측 시간 — 원본 데이터 → 피처 변환 → 확률
OOF Macro F1: 전체 Train 5-fold OOF (experiments/preds 저장 예측). 앙상블은 세 모델 확률 1/3씩 평균

In [9]:
# 모델별 크기·예측 시간 (원본 데이터 → 피처 변환 → 확률, 1명 20회·500명 5회 중앙값)
X_one, X_batch = X_timing.iloc[:1], X_timing

def predict_ensemble(Xr):
    """최종 3개 앙상블: 확률 1/3씩 평균."""
    return sum(m.predict_proba(Xr) for m in fitted.values()) / len(fitted)

TARGETS = {   # 이름: (전체 Train OOF Macro F1, 예측 함수, 저장할 모델)
    "LR (E + 위치 구간)": (0.5043, fitted["lr_E_position"].predict_proba, (fitted["lr_E_position"],)),
    "가중치 XGBoost (E)": (0.4954, fitted["xgb_E_weighted"].predict_proba, (fitted["xgb_E_weighted"],)),
    "LightGBM g3": (0.5107, fitted["lgbm_g3"].predict_proba, (fitted["lgbm_g3"],)),
    "최종 3개 앙상블 (1:1:1)": (0.5284, predict_ensemble, tuple(fitted.values())),
}

rows = []
for name, (oof, predict, parts) in TARGETS.items():
    rows.append({"모델": name, "OOF Macro F1": oof, "크기(MB)": size_mb(*parts),
                 "1명(ms)": median_ms(lambda: predict(X_one), 20),
                 "500명(ms)": median_ms(lambda: predict(X_batch), 5)})
final_table = pd.DataFrame(rows)
print(final_table.to_string(index=False))

               모델  OOF Macro F1  크기(MB)   1명(ms)  500명(ms)
   LR (E + 위치 구간)        0.5043    3.72 1423.787  5379.192
  가중치 XGBoost (E)        0.4954    7.07 1572.674  4242.062
      LightGBM g3        0.5107   37.78 1584.050  7308.421
최종 3개 앙상블 (1:1:1)        0.5284   48.57 4249.300 15228.379


## 5. 피처 변환 vs 모델 계산 (1명 기준)

In [10]:
# 피처 변환 vs 모델 계산 나눠 보기 (1명 기준)
split_rows = []
for key, label in [("lr_E_position", "LR (E + 위치 구간)"), ("xgb_E_weighted", "가중치 XGBoost (E)"), ("lgbm_g3", "LightGBM g3")]:
    pipe = fitted[key]
    feat, model = pipe.named_steps["features"], pipe.named_steps["model"]
    Z_one = feat.transform(X_one)
    split_rows.append({"모델": label,
                       "피처 변환(ms)": median_ms(lambda: feat.transform(X_one), 20),
                       "모델 계산(ms)": median_ms(lambda: model.predict_proba(Z_one), 20)})
split_table = pd.DataFrame(split_rows)
print(split_table.to_string(index=False))

             모델  피처 변환(ms)  모델 계산(ms)
 LR (E + 위치 구간)   1072.958      0.061
가중치 XGBoost (E)   1144.168    141.257
    LightGBM g3   1384.894      7.243


## 6. 요약 — LR 단독 vs 최종 3개 앙상블

In [ ]:
# 요약 — LR 단독 vs 최종 3개 앙상블
lr, ens = final_table.iloc[0], final_table.iloc[3]
print("성능 차이(OOF):", round(ens["OOF Macro F1"] - lr["OOF Macro F1"], 4))
print("크기:", round(ens["크기(MB)"] / lr["크기(MB)"], 1), "배")
print("1명 예측:", round(ens["1명(ms)"] / lr["1명(ms)"], 1), "배")
print("500명 예측:", round(ens["500명(ms)"] / lr["500명(ms)"], 1), "배")

성능 차이(OOF): 0.0241
크기: 13.1 배
1명 예측: 4.3 배
500명 예측: 3.4 배


### 6-1. 결론 (Mac 실행 기준, 시간은 2회 측정 범위)
- 성능: 최종 3개 앙상블 OOF 0.5284, LR 단독 0.5043 → 차이 0.0241 → 서비스 기획 채택 기준(0.02 이하) 초과 → 서비스 모델은 최종 3개 앙상블
- 비용: 앙상블이 LR 대비 크기 13.1배 (48.57 MB vs 3.72 MB), 1명 예측 약 3~4배 (약 4.0~4.2초 vs 0.9~1.4초)
- 1명 예측 시간의 대부분은 피처 변환 (LR: 변환 약 1.1~1.2초 / 모델 0.07 ms) → 속도 개선은 모델보다 피처 변환 쪽
- 측정 시간은 실행마다 30~50% 흔들림 (1초 단위 측정) → 크기·배수 중심으로 해석
- LR 단독은 저사양·빠른 1차 결과용 대안 (확장 아이디어, 검증 안 됨)